# Deep Learning Project Template (PyTorch)

**Purpose:** A fully-commented, reusable skeleton for any PyTorch deep learning project.  
Copy this notebook, plug in your Dataset and model architecture, and follow the steps.

---

## Real-World Analogy

Training a neural network is like **coaching a sports team**:
- **Dataset / DataLoader** = players showing up to practice every day
- **Model** = the playbook (architecture determines strategy)
- **Loss function** = the scoreboard — tells you how far you are from winning
- **Optimizer** = the coach adjusting tactics based on the scoreboard
- **Epochs** = seasons — each one the team practices the whole playbook
- **Validation** = exhibition games — test without stakes to measure real progress
- **Checkpointing** = saving the best season's game tape

---

## Template Steps
1. Imports & Configuration
2. Dataset & DataLoader
3. Model Architecture
4. Weight Initialisation
5. Loss, Optimizer & Scheduler
6. Training Loop
7. Evaluation & Per-Class Metrics
8. Export & Serving

## Installation
```bash
pip install torch torchvision matplotlib fastapi uvicorn pillow python-multipart
```

In [ ]:
# ============================================================
# STEP 1 — IMPORTS & CONFIGURATION
# ============================================================

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100
import random, os, time
from datetime import datetime
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

try:
    import torchvision
    from torchvision import transforms, datasets
    TV_AVAILABLE = True
    print('torchvision available')
except ImportError:
    TV_AVAILABLE = False
    print('torchvision not installed — using synthetic data')

# ── Device selection ───────────────────────────────────────
# MPS = Apple Silicon GPU, CUDA = NVIDIA GPU, else CPU
if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')
print(f'Using device: {DEVICE}')

# ── Reproducibility ────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── USER CONFIGURATION ─────────────────────────────────────
NUM_CLASSES    = 10          # ← number of output classes
BATCH_SIZE     = 64
LEARNING_RATE  = 1e-3
NUM_EPOCHS     = 10
PATIENCE       = 5           # early stopping patience
CHECKPOINT_DIR = Path('checkpoints')
CHECKPOINT_DIR.mkdir(exist_ok=True)

print('\nConfiguration ready.')

In [ ]:
# ============================================================
# STEP 2 — DATASET & DATALOADER
# ============================================================
# PyTorch separates DATA (Dataset) from ITERATION (DataLoader).
#
# Dataset  = knows how to get one example (like a library catalogue)
# DataLoader = batches + shuffles + parallelises (like a librarian)
#
# To use your own data, inherit from Dataset and implement:
#   __len__(self)         → total number of examples
#   __getitem__(self, i)  → (features, label) for index i

# ── Option A: torchvision built-in datasets (MNIST / CIFAR10) ─
if TV_AVAILABLE:
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))   # MNIST mean/std
    ])
    train_dataset = datasets.MNIST(root='./data', train=True,
                                    download=True, transform=transform)
    test_dataset  = datasets.MNIST(root='./data', train=False,
                                    download=True, transform=transform)
    # Split train into train + validation
    n_val   = int(0.15 * len(train_dataset))
    n_train = len(train_dataset) - n_val
    train_dataset, val_dataset = random_split(
        train_dataset, [n_train, n_val],
        generator=torch.Generator().manual_seed(SEED)
    )
    IN_CHANNELS = 1
    print(f'MNIST loaded — train={n_train:,} val={n_val:,} test={len(test_dataset):,}')

else:
    # ── Option B: Custom synthetic Dataset ────────────────────
    class SyntheticImageDataset(Dataset):
        """Generates random 28×28 single-channel images with random labels."""
        def __init__(self, n_samples=5000, n_classes=10):
            self.X = torch.randn(n_samples, 1, 28, 28)
            self.y = torch.randint(0, n_classes, (n_samples,))

        def __len__(self):
            return len(self.y)

        def __getitem__(self, idx):
            return self.X[idx], self.y[idx]

    full_ds  = SyntheticImageDataset(n_samples=6000, n_classes=NUM_CLASSES)
    n_train  = 4200
    n_val    = 900
    n_test   = 900
    train_dataset, val_dataset, test_dataset = random_split(
        full_ds, [n_train, n_val, n_test],
        generator=torch.Generator().manual_seed(SEED)
    )
    IN_CHANNELS = 1
    print(f'Synthetic dataset — train={n_train} val={n_val} test={n_test}')

# ── DataLoaders ────────────────────────────────────────────
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                           shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                           shuffle=False, num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                           shuffle=False, num_workers=0, pin_memory=False)

# Visualise a batch
imgs, lbls = next(iter(train_loader))
print(f'Batch shape: {imgs.shape}  Labels: {lbls[:8].tolist()}')

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, img, lbl in zip(axes, imgs[:8], lbls[:8]):
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(str(lbl.item()), fontsize=8)
    ax.axis('off')
plt.suptitle('Sample Training Images')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 3 — MODEL ARCHITECTURE
# ============================================================
# Replace this with your architecture. The template provides
# a CNN suited for 28×28 grayscale images (MNIST-style).
#
# Architecture principles:
#   BatchNorm  → stabilises training, allows higher LR
#   Dropout    → regularisation (prevents overfitting)
#   MaxPool    → reduces spatial size, captures hierarchy

class TemplateCNN(nn.Module):
    """
    Two convolutional blocks followed by a fully-connected classifier.
    Input : (B, C, 28, 28)
    Output: (B, num_classes) — raw logits
    """
    def __init__(self, in_channels=1, num_classes=10):
        super().__init__()

        # ── Block 1: 1→32 channels ─────────────────────────
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),          # 28×28 → 14×14
            nn.Dropout2d(0.25),
        )

        # ── Block 2: 32→64 channels ────────────────────────
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),          # 14×14 → 7×7
            nn.Dropout2d(0.25),
        )

        # ── Classifier ─────────────────────────────────────
        self.classifier = nn.Sequential(
            nn.Flatten(),                # 64 × 7 × 7 = 3136
            nn.Linear(64 * 7 * 7, 256, bias=False),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.classifier(x)
        return x   # raw logits — loss fn applies softmax internally


model = TemplateCNN(in_channels=IN_CHANNELS, num_classes=NUM_CLASSES).to(DEVICE)

# Parameter count
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

# Quick forward pass sanity check
with torch.no_grad():
    dummy = torch.zeros(2, IN_CHANNELS, 28, 28).to(DEVICE)
    out   = model(dummy)
    print(f'Output shape: {out.shape}  (expected: [2, {NUM_CLASSES}])')

In [ ]:
# ============================================================
# STEP 4 — WEIGHT INITIALISATION
# ============================================================
# Poor initialisation → vanishing/exploding gradients on layer 1.
# Rules of thumb:
#   Conv / Linear with ReLU → Kaiming (He) initialisation
#   Conv / Linear with Tanh  → Xavier (Glorot) initialisation
#   BatchNorm                → weight=1, bias=0 (already default)

def init_weights(module):
    """Apply sensible default initialisations to a model."""
    if isinstance(module, nn.Conv2d):
        nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Linear):
        nn.init.xavier_uniform_(module.weight)
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, (nn.BatchNorm2d, nn.BatchNorm1d)):
        nn.init.ones_(module.weight)
        nn.init.zeros_(module.bias)

model.apply(init_weights)
print('Weight initialisation applied (Kaiming for conv, Xavier for linear).')

In [ ]:
# ============================================================
# STEP 5 — LOSS, OPTIMIZER & SCHEDULER
# ============================================================
# Loss function:
#   Classification → CrossEntropyLoss (includes softmax internally)
#   Regression     → MSELoss or HuberLoss
#   label_smoothing prevents over-confident predictions
#
# Optimizer:
#   AdamW = Adam with decoupled weight decay — almost always
#   a good default for deep learning.
#
# Scheduler:
#   OneCycleLR = warmup → peak → cosine decay in one epoch sweep
#   Often beats step or exponential decay.

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4
)

scheduler = OneCycleLR(
    optimizer,
    max_lr=LEARNING_RATE,
    steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS,
    pct_start=0.3,       # 30% of steps = warmup
    anneal_strategy='cos'
)

print(f'Loss      : CrossEntropyLoss(label_smoothing=0.1)')
print(f'Optimizer : AdamW  lr={LEARNING_RATE}  wd=1e-4')
print(f'Scheduler : OneCycleLR  ({NUM_EPOCHS} epochs, {len(train_loader)} steps/epoch)')

In [ ]:
# ============================================================
# STEP 6 — TRAINING LOOP
# ============================================================
# Every DL training loop follows this pattern:
#
#   for each epoch:
#     TRAIN:  forward → loss → backward → clip grads → step
#     VALID:  forward only (no gradients) → record metrics
#     checkpoint if val loss improved
#     early stop if no improvement for PATIENCE epochs

def train_one_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()   # enables Dropout + BatchNorm training mode
    total_loss, correct, total = 0.0, 0, 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad(set_to_none=True)   # faster than zero_grad()
        logits = model(X)
        loss   = criterion(logits, y)
        loss.backward()

        # Gradient clipping: prevents exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()   # OneCycleLR steps per batch, not per epoch

        total_loss += loss.item() * X.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total   += X.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()   # disables Dropout, uses running stats for BatchNorm
    total_loss, correct, total = 0.0, 0, 0

    for X, y in loader:
        X, y   = X.to(device), y.to(device)
        logits = model(X)
        loss   = criterion(logits, y)

        total_loss += loss.item() * X.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total   += X.size(0)

    return total_loss / total, correct / total


# ── Training loop ──────────────────────────────────────────
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss = float('inf')
patience_counter = 0
best_epoch = 0

print(f'Training for up to {NUM_EPOCHS} epochs (early stop patience={PATIENCE})\n')
print(f'{"Epoch":>5}  {"Train Loss":>10}  {"Train Acc":>9}  {"Val Loss":>8}  {"Val Acc":>7}')
print('-' * 55)

for epoch in range(1, NUM_EPOCHS + 1):
    t_start = time.time()

    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer,
                                       scheduler, criterion, DEVICE)
    vl_loss, vl_acc = evaluate(model, val_loader, criterion, DEVICE)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    elapsed = time.time() - t_start
    print(f'{epoch:>5}  {tr_loss:>10.4f}  {tr_acc:>8.1%}  {vl_loss:>8.4f}  {vl_acc:>6.1%}  ({elapsed:.1f}s)')

    # Checkpointing
    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        best_epoch    = epoch
        patience_counter = 0
        torch.save({
            'epoch'      : epoch,
            'model_state': model.state_dict(),
            'optim_state': optimizer.state_dict(),
            'val_loss'   : vl_loss,
            'val_acc'    : vl_acc,
        }, CHECKPOINT_DIR / 'best_model.pt')
        print(f'       ↑ New best saved (val_loss={vl_loss:.4f})')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'\nEarly stop at epoch {epoch} (no improvement for {PATIENCE} epochs)')
            break

print(f'\nBest epoch: {best_epoch}  Best val loss: {best_val_loss:.4f}')

In [ ]:
# ── Training curves ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs_ran = range(1, len(history['train_loss']) + 1)
axes[0].plot(epochs_ran, history['train_loss'], label='Train', color='steelblue')
axes[0].plot(epochs_ran, history['val_loss'],   label='Val',   color='tomato')
axes[0].axvline(best_epoch, linestyle='--', color='green', alpha=0.7, label=f'Best (epoch {best_epoch})')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()

axes[1].plot(epochs_ran, [a*100 for a in history['train_acc']], label='Train', color='steelblue')
axes[1].plot(epochs_ran, [a*100 for a in history['val_acc']],   label='Val',   color='tomato')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training & Validation Accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 7 — EVALUATION & PER-CLASS METRICS
# ============================================================
# Load the BEST checkpoint before evaluating on the test set.
# Never evaluate on test during training — the test set is
# your final "exam" taken once.

checkpoint = torch.load(CHECKPOINT_DIR / 'best_model.pt', map_location=DEVICE)
model.load_state_dict(checkpoint['model_state'])
print(f'Loaded best checkpoint from epoch {checkpoint["epoch"]}  (val_loss={checkpoint["val_loss"]:.4f})')

# Collect all predictions
all_preds, all_labels = [], []
model.eval()
with torch.no_grad():
    for X, y in test_loader:
        logits = model(X.to(DEVICE))
        preds  = logits.argmax(dim=1).cpu()
        all_preds.append(preds)
        all_labels.append(y)

all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

test_acc = (all_preds == all_labels).mean()
print(f'\nTest Accuracy: {test_acc:.1%}')

# Per-class accuracy
print('\nPer-class accuracy:')
for cls in range(NUM_CLASSES):
    mask   = all_labels == cls
    cls_acc = (all_preds[mask] == cls).mean() if mask.sum() > 0 else 0.0
    bar    = '█' * int(cls_acc * 20)
    print(f'  Class {cls}: {cls_acc:6.1%}  {bar}')

# Confusion matrix (normalised)
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(8, 7))
plt.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.colorbar()
plt.title('Normalised Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 8 — EXPORT & SERVING
# ============================================================
# TorchScript converts a PyTorch model into a static graph
# that runs without the Python runtime — needed for C++
# deployment, mobile, or ONNX export.

model.eval()

# ── TorchScript export ─────────────────────────────────────
try:
    scripted = torch.jit.script(model)
    script_path = CHECKPOINT_DIR / 'model_scripted.pt'
    scripted.save(str(script_path))
    print(f'TorchScript saved: {script_path}')

    # Verify
    loaded_script = torch.jit.load(str(script_path))
    dummy = torch.zeros(1, IN_CHANNELS, 28, 28)
    with torch.no_grad():
        out1 = model(dummy)
        out2 = loaded_script(dummy)
    assert torch.allclose(out1, out2), 'TorchScript output mismatch!'
    print('TorchScript verification PASSED.')
except Exception as e:
    print(f'TorchScript failed ({e}). Try torch.jit.trace() for non-dynamic models.')

# ── ONNX export (optional) ─────────────────────────────────
try:
    onnx_path = CHECKPOINT_DIR / 'model.onnx'
    dummy = torch.zeros(1, IN_CHANNELS, 28, 28)
    torch.onnx.export(
        model, dummy, str(onnx_path),
        input_names=['image'], output_names=['logits'],
        dynamic_axes={'image': {0: 'batch_size'}, 'logits': {0: 'batch_size'}},
        opset_version=17
    )
    print(f'ONNX model saved: {onnx_path}')
except Exception as e:
    print(f'ONNX export skipped ({e})')

In [ ]:
# ── FastAPI image serving code — save as app.py ───────────
fastapi_code = '''
# app.py — Deep Learning Model Serving
# Run: uvicorn app:app --reload

from fastapi import FastAPI, UploadFile, File, HTTPException
from pydantic import BaseModel
import torch, io
from PIL import Image
from torchvision import transforms

app = FastAPI(title="DL Image Classifier", version="1.0")

# Load model at startup
model = torch.jit.load("checkpoints/model_scripted.pt", map_location="cpu")
model.eval()

# Must match training transforms
transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

class PredictResponse(BaseModel):
    predicted_class: int
    confidence: float

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict", response_model=PredictResponse)
async def predict(file: UploadFile = File(...)):
    if not file.content_type.startswith("image/"):
        raise HTTPException(400, "File must be an image")
    img_bytes = await file.read()
    img = Image.open(io.BytesIO(img_bytes)).convert("L")
    tensor = transform(img).unsqueeze(0)   # add batch dimension
    with torch.no_grad():
        logits = model(tensor)
        probs  = torch.softmax(logits, dim=1)[0]
        cls    = probs.argmax().item()
        conf   = probs[cls].item()
    return PredictResponse(predicted_class=cls, confidence=round(conf, 4))
'''
print(fastapi_code)

In [ ]:
# ── DL Debugging Checklist ─────────────────────────────────
checklist = [
    ('Loss stays at log(N)',       'Weights not initialised / LR too low / labels wrong'),
    ('Loss explodes (NaN/inf)',    'LR too high → add gradient clipping (max_norm=1.0)'),
    ('Train acc high, val low',    'Overfitting → add Dropout, augmentation, weight decay, less capacity'),
    ('Val acc stops improving',    'LR too low → try LR warmup or cosine annealing'),
    ('GPU utilisation < 50%',      'DataLoader bottleneck → increase num_workers, use pin_memory=True'),
    ('Slow on Apple Silicon',      'Use device="mps" and torch.backends.mps.is_available()'),
    ('Outputs all same class',     'Class imbalance → use weighted loss or balanced sampler'),
]

print('DL Debugging Checklist')
print('=' * 70)
for symptom, fix in checklist:
    print(f'\nSymptom : {symptom}')
    print(f'Fix     : {fix}')

## Interview Questions & Answers

---

**Q1: Why do we call `model.train()` and `model.eval()`?**

A: Two layers behave differently during training vs inference:
- **Dropout** — randomly zeros activations during training (regularisation), but passes all activations during eval to get deterministic predictions.
- **BatchNorm** — during training uses the *batch* mean/variance; during eval uses *running* mean/variance accumulated over all training batches.
Forgetting `.eval()` before validation is a very common bug — your validation loss will appear better than it really is because Dropout is still active.

---

**Q2: What is gradient clipping and why is it needed?**

A: Gradient clipping caps the L2 norm of the gradient vector at `max_norm`. Without it, a single bad batch (e.g., very high loss due to an outlier) can produce enormously large gradients, causing a parameter update so large the model "jumps" to a bad region — the loss spikes to NaN and training collapses. This is called an **exploding gradient**. Clipping rescales the gradient if its norm exceeds the cap, leaving the direction unchanged. `max_norm=1.0` is a safe default for most architectures.

---

**Q3: What is OneCycleLR and why does it often outperform a fixed learning rate?**

A: OneCycleLR (Leslie Smith, 2018) varies the learning rate in one cycle: linearly warm up to `max_lr`, then cosine-anneal back to `max_lr / div_factor`. The warm-up prevents early instability when random weights produce high-variance gradients. The high LR phase explores the loss landscape broadly (escaping shallow local minima). The cool-down fine-tunes into a sharp minimum. It consistently converges faster than constant LR and often finds better final accuracy. Key parameters: `max_lr`, `pct_start` (fraction of steps spent warming up).

---

**Q4: What is the difference between `torch.jit.script` and `torch.jit.trace`?**

A: Both convert PyTorch to TorchScript for C++ / mobile deployment.
- **`torch.jit.script`** — parses the Python source code. Preserves all control flow (if/for/while). Works on any model. Fails if your code uses unsupported Python.
- **`torch.jit.trace`** — records a *single forward pass* with dummy input. Control flow is baked in (if you trace with `batch_size=1`, if-branches on batch size won't work at other sizes). Easier to use but fragile with dynamic shapes.
Use `script` when your model has data-dependent control flow; `trace` for simpler, static-graph models.

---

**Q5: What is label smoothing and why does it help?**

A: Standard CrossEntropyLoss trains the model to predict probability 1.0 for the correct class and 0.0 for all others. This pushes logits to ±∞, making the model **overconfident** and brittle. Label smoothing with ε=0.1 distributes ε/(K-1) of the probability to non-target classes, so the target is 0.9 instead of 1.0. This acts as regularisation, typically improving generalisation by 0.5–2% on test sets. It also helps the model output better-calibrated probabilities.

---

**Q6: Explain early stopping and why you checkpoint the best model rather than the last.**

A: As training continues past the optimal point, the model starts memorising training data (overfitting) — training loss keeps falling but validation loss rises. Early stopping halts training when validation loss hasn't improved for `patience` epochs. The `patience` hyperparameter gives the optimizer a chance to escape temporary plateaus without stopping too early. We save the checkpoint at the *lowest validation loss epoch* because that's the weight configuration that generalises best — the last epoch's weights are more overfit.

## Recommended Resources

| Resource | Link | Why |
|---|---|---|
| PyTorch Official Tutorials | https://pytorch.org/tutorials/ | Best starting point |
| fast.ai Deep Learning Course | https://course.fast.ai/ | Best practical DL course (free) |
| Andrej Karpathy — makemore | https://github.com/karpathy/makemore | Learn from the master |
| OneCycleLR Paper | https://arxiv.org/abs/1708.07120 | Leslie Smith 2018 |
| Label Smoothing Paper | https://arxiv.org/abs/1512.00567 | Szegedy et al. (Inception) |
| PyTorch Profiler | https://pytorch.org/tutorials/recipes/recipes/profiler_recipe.html | GPU utilisation debugging |

---
*Template v1.0 — copy, add your Dataset and model, and train.*